# 第11回：回帰モデルの基礎と練習

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar1_Public/blob/main/notebooks/session09_10_regression_classification.ipynb)


**DSゼミナールⅠ（2026年度）**
熊本大学 データサイエンス学科

回帰は、連続値を予測するための機械学習手法です。今回は回帰の基本を学び、評価方法を確認したうえで、実際に手を動かす練習を行います。

---

## 📋 本日の目標

1. 回帰と分類の違いを説明できる
2. 線形回帰モデルの仕組みを理解できる
3. 回帰評価指標の意味を説明できる
4. データを分割して回帰モデルを構築できる
5. 予測と実測の比較を可視化できる

---

## 1. 回帰とは

回帰は、説明変数（特徴量）から目的変数として連続値を予測する問題です。代表的な例としては、住宅価格予測、売上予測、気温予測などがあります。

### 1-1. 回帰と分類の違い

| 種類 | 目的変数 | 例 |
| ---- | ---- | ---- |
| 分類 | 離散ラベル | 画像分類、心疾患の有無予測 |
| 回帰 | 連続値 | 価格予測、体重予測、売上の予測 |

### 1-2. 線形回帰の基本式

線形回帰モデルは次のように表現されます。

`y = w_0 + w_1 x_1 + w_2 x_2 + ... + w_p x_p + ε`

ここで、

- `y` = 予測される連続値
- `x_i` = 各特徴量
- `w_i` = 重み（係数）
- `ε` = 誤差項

重み `w_i` が大きいほど、その特徴量の影響は大きくなります。

### 1-3. 線形回帰の学習と予測

線形回帰モデルを学習するとは、訓練データに対して予測誤差を最小化する `w_0, w_1, ..., w_p` を見つけることです。

- `fit(X, y)` によって重み `coef_` と切片 `intercept_` を求める
- 一般的な手法は最小二乗法で、残差の二乗和を最小にする
- 残差 `y_i - ŷ_i` は、各サンプルの実測値と予測値の差

線形回帰の学習目標は次の式で表せます。

`min_w Σ_{i=1}^n (y_i - ŷ_i)^2`

この式は、予測値と実測値の差を二乗して合計した値をできるだけ小さくすることを意味します。

### 1-4. 線形回帰モデルの特徴

- 単純で計算が高速
- 特徴量と目的変数の関係が線形だと仮定する
- 訓練データ内のノイズも学習してしまうと過学習になる
- 多重共線性や外れ値があると係数の推定が不安定になる

### 1-5. 代表的な回帰モデル

#### 1-5-1. 線形回帰（LinearRegression）

- 説明変数の線形結合で目的変数を予測する基本モデル
- 重み `coef_` が解釈しやすい
- 特徴量のスケールが異なる場合は標準化が必要になることがある

#### 1-5-2. リッジ回帰（Ridge）

- 線形回帰に `L2` 正則化を加えたモデル
- 係数の大きさを抑えて過学習を防ぐ
- 特徴量の共線性がある場合に安定した推定ができる

#### 1-5-3. ラッソ回帰（Lasso）

- 線形回帰に `L1` 正則化を加えたモデル
- 重要でない特徴量の係数をゼロにすることで特徴選択効果がある
- 変数選択と予測精度のバランスを取るのに useful

#### 1-5-4. サポートベクター回帰（SVR）

- サポートベクターマシンを回帰問題に拡張したモデル
- 誤差がある程度小さい範囲内では損失をゼロとみなす `ε`-insensitive loss を使う
- カーネル（線形、RBF、多項式）を使うことで非線形な関係を学習できる
- 予測性能が高い反面、計算量が増える場合がある

#### 1-5-5. 決定木回帰（DecisionTreeRegressor）

- データを分割しながら予測値を決定する非線形モデル
- 特徴量のスケール調整が不要
- 過学習しやすいので深さ制限や最小サンプル数の調整が必要

#### 1-5-6. ランダムフォレスト回帰（RandomForestRegressor）

- 複数の決定木を組み合わせたアンサンブルモデル
- 単一の決定木よりも過学習に強く、性能が安定しやすい
- 特徴量の重要度も確認できる

---

#### 図: 回帰モデルの系統図

```mermaid
graph LR
    A[線形モデル]
    B[正則化モデル]
    C[カーネル/非線形]
    D[決定木系]
    E[アンサンブル]
    A -->|例: LinearRegression| B
    B -->|Ridge / Lasso| C
    C -->|SVR (RBF)など| D
    D -->|DecisionTreeRegressor| E
    E -->|RandomForest| E
```

### 1-6. 回帰の評価で重要な観点

回帰モデルでは、単に誤差を小さくするだけでなく、次の点を確認します。

- 予測誤差の大きさはどれくらいか
- 予測値と実測値の関係はどのようになっているか
- 誤差が特定の値域で大きくならないか
- モデルが系統的に過大/過小評価していないか

---

## 2. 回帰評価指標

回帰モデルの評価では、予測値と実測値の差を数値化してモデルの良し悪しを判断します。評価指標は、用途や目的に応じて使い分けることが重要です。

### 2-1. MSE（平均二乗誤差）

`MSE = (1/n) Σ_{i=1}^n (y_i - ŷ_i)^2`

- 予測誤差を二乗して平均した値
- 大きな誤差をより強く罰する
- 学習時の目的関数（損失関数）としても使われることが多い

> MSEは外れ値の影響を大きく受けるため、外れ値があるデータでは注意が必要です。

### 2-2. RMSE（平方根平均二乗誤差）

`RMSE = sqrt(MSE)`

- 誤差を元の目的変数の単位に戻した指標
- 単位が元データと同じなので直感的に理解しやすい
- 例：目的変数が住宅価格なら、「万円あたりの誤差」として読み取れる

### 2-3. MAE（平均絶対誤差）

`MAE = (1/n) Σ_{i=1}^n |y_i - ŷ_i|`

- 誤差の絶対値を平均した値
- 外れ値の影響がMSEより小さい
- 誤差の大きさを対称的に評価する

> MAEは「平均してどれだけズレるか」を直感的に示すため、実務評価で使いやすい場合があります。

### 2-4. R²（決定係数）

`R^2 = 1 - Σ_i (y_i - ŷ_i)^2 / Σ_i (y_i - ȳ)^2`

- モデルが目的変数の変動をどれだけ説明できているかを示す
- 1に近いほど良い
- 0以下になることもあり、その場合は単純平均 `ȳ` よりも悪い予測になる

### 2-5. 指標の使い分け

- `MSE` / `RMSE`：大きな誤差を特に嫌う場合、また学習時の損失関数として利用する場合
- `MAE`：外れ値に強く、誤差の平均的な大きさを知りたい場合
- `R²`：データの分散に対する説明力を見たい場合

### 2-6. 評価時の注意点

- 同じデータセットと同じ分割方法で比較する
- `RMSE` や `MAE` は目的変数のスケールに依存するため、異なるデータセット間の直接比較は避ける
- 訓練データとテストデータでの評価値を比較し、過学習・未学習を判断する
- 残差プロットやヒストグラムも合わせて確認し、予測誤差の分布を把握する

#### 評価プロセスの図示

```mermaid
graph TD
    D[データ] --> M[モデル学習]
    M --> P[予測値]
    P --> R[残差計算]
    R --> V1[RMSE / MAE]
    R --> V2[残差プロット / ヒストグラム]
    P --> V3[実測 vs 予測散布図 or Bland-Altman]
```

---

## 3. 回帰分析のVisualization

### 3-1. 実測値 vs 予測値の散布図
実測値と予測値の散布図は、モデルの全体的な予測精度を直感的に把握するための基本的な可視化です。理想的には点が45度線の周りに集まり、系統的な偏りがないことが望まれます。

- 横軸：実測値 `y` 
- 縦軸：予測値 `ŷ`
- 45度線を描くことで、過大/過小評価の傾向を確認できる

#### 実装例


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ダミーデータを生成
np.random.seed(42)
y_test = np.linspace(100, 300, 50) + np.random.normal(0, 10, 50)
y_pred = y_test + np.random.normal(0, 15, 50)  # 若干の予測誤差を含む

# 散布図
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.7, s=50)

# 45度線を描画
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='完全予測ライン')

plt.xlabel('実測値 (y)', fontsize=12)
plt.ylabel('予測値 (ŷ)', fontsize=12)
plt.title('実測値 vs 予測値', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.tight_layout()
plt.show()


**グラフの読み取り：**
- 点が赤い45度線に近い → 予測精度が高い
- 点が45度線より上 → 予測値が過大評価（予測値 > 実測値）
- 点が45度線より下 → 予測値が過小評価（予測値 < 実測値）

### 3-2. Bland-Altman プロット
Bland-Altman プロットは、各サンプルの平均予測値と差（実測値 - 予測値）を表示し、系統誤差や誤差の広がりを確認するために有効です。

- 横軸：実測値と予測値の平均 `(y + ŷ)/2`
- 縦軸：差 `y - ŷ` （実測値 - 予測値）
- 中央線は平均差（bias）
- 上下に標準偏差 ±1.96σ を表示すると、95%信頼範囲の目安になる

#### 実装例


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ダミーデータ（上記と同じ）
np.random.seed(42)
y_test = np.linspace(100, 300, 50) + np.random.normal(0, 10, 50)
y_pred = y_test + np.random.normal(0, 15, 50)

# 平均値と差を計算
mean_val = (y_test + y_pred) / 2
diff = y_test - y_pred

# バイアスと信頼限界
mean_diff = np.mean(diff)
std_diff = np.std(diff, ddof=1)
upper_limit = mean_diff + 1.96 * std_diff
lower_limit = mean_diff - 1.96 * std_diff

# プロット
plt.figure(figsize=(8, 6))
plt.scatter(mean_val, diff, alpha=0.7, s=50)
plt.axhline(mean_diff, color='red', linestyle='-', lw=2, label=f'平均差 = {mean_diff:.2f}')
plt.axhline(upper_limit, color='gray', linestyle='--', lw=1.5, label=f'±1.96σ (n={len(diff)})')
plt.axhline(lower_limit, color='gray', linestyle='--', lw=1.5)

plt.xlabel('(実測値 + 予測値) / 2', fontsize=12)
plt.ylabel('実測値 - 予測値', fontsize=12)
plt.title('Bland-Altman プロット', fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


**グラフの読み取り：**
- 中央線が0に近い → バイアス（系統誤差）が小さい
- 点が上下の破線内に均等に分布 → 予測値が均一に分布している
- 点が上下いずれかに集中 → 特定の値域で系統的な偏りがある

### 3-3. 残差のヒストグラム / 箱ひげ図
残差のヒストグラムと箱ひげ図は、予測誤差の分布や外れ値の存在を把握するのに役立ちます。残差が正規分布に近く、平均が0に近いほどモデルの仮定に合っています。

- ヒストグラム：残差の分布を確認する
- 箱ひげ図：外れ値や分布の偏りを確認する
- 理想：残差が左右対称で中央値が0付近

#### 実装例


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ダミーデータ（上記と同じ）
np.random.seed(42)
y_test = np.linspace(100, 300, 50) + np.random.normal(0, 10, 50)
y_pred = y_test + np.random.normal(0, 15, 50)

# 残差を計算
residuals = y_test - y_pred

# 2つのサブプロットを作成
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左図：ヒストグラム
axes[0].hist(residuals, bins=15, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].axvline(np.mean(residuals), color='red', linestyle='--', lw=2, label=f'平均 = {np.mean(residuals):.2f}')
axes[0].axvline(0, color='green', linestyle='--', lw=2, label='0（理想値）')
axes[0].set_xlabel('残差', fontsize=12)
axes[0].set_ylabel('頻度', fontsize=12)
axes[0].set_title('残差のヒストグラム', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# 右図：箱ひげ図
axes[1].boxplot(residuals, vert=True, patch_artist=True, 
                boxprops=dict(facecolor='lightblue', alpha=0.7),
                medianprops=dict(color='red', lw=2))
axes[1].axhline(0, color='green', linestyle='--', lw=2, label='0（理想値）')
axes[1].set_ylabel('残差', fontsize=12)
axes[1].set_title('残差の箱ひげ図', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_xticklabels(['残差'])

plt.tight_layout()
plt.show()


**グラフの読み取り：**
- ヒストグラムが0を中心に左右対称 → 正規分布に近い（良い）
- 箱ひげ図の中央線が0に近い → バイアスが小さい
- 箱ひげ図の外側に点がある → 外れ値が存在する可能性
- ヒストグラムが0から大きく離れている → モデルの系統誤差がある

---

## 4. 練習課題：線形回帰を実装しよう

以下の手順で回帰モデルを構築し、評価してください。

### 3-1. 演習データの準備

今回はScikit-learn付属の糖尿病データセットを使います。

- `load_diabetes()` は特徴量と連続値の目的変数を含む標準データセットです
- 目的変数は疾患進行度の数値です

### 3-2. 実装手順

1. `sklearn.datasets.load_diabetes` でデータを読み込む
2. `train_test_split` で学習データとテストデータに分ける
3. 次の3つのモデルを比較する
   - `LinearRegression`
   - `SVR`
   - `RandomForestRegressor`
4. 各モデルに対して `GridSearchCV` を使い、パラメータチューニングを行う
5. テストデータで予測を行い、`MSE`, `RMSE`, `MAE`, `R2` を計算する
6. 実測値 vs 予測値の散布図、残差のヒストグラム、Bland-Altman プロットを作成する
7. 3モデルの性能を比較し、最も適したモデルを選ぶ

### 3-3. データ説明

**糖尿病データセット（Diabetes Dataset）について**

今回の練習で使用する糖尿病データセットは、Scikit-learn の標準データセットです。以下のような特徴があります：

- **データ数**：442サンプル
- **特徴量**：10個
  - `age`：患者の年齢
  - `sex`：患者の性別
  - `bmi`：体質指数（Body Mass Index）
  - `bp`：平均血圧
  - `s1-s6`：血清測定値（6種類）
- **目的変数**：1年後の疾患進行度（連続値、0～約345の範囲）
- **用途**：回帰モデルの学習・評価の基本例として最適

このデータセットは比較的シンプルで、複数の連続値の特徴量から1つの連続値を予測する典型的な回帰問題です。特に以下の点で学習に適しています：

- 特徴量が少ないため、パラメータチューニングが扱いやすい
- ノイズが程度よく含まれており、現実的な学習課題
- 線形・非線形モデル両方の特性を観察できる

### 3-4. コード例

#### 日本語表示の準備（Colab向け）


In [ ]:
import sys
import subprocess
import matplotlib.pyplot as plt

try:
    import japanize_matplotlib
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "japanize-matplotlib", "-q"])
    import japanize_matplotlib

plt.rcParams["font.family"] = "IPAexGothic"
plt.rcParams["axes.unicode_minus"] = False


#### モデル比較とパラメータチューニングの骨組み


In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import matplotlib.pyplot as plt

# データ読み込み
X, y = load_diabetes(return_X_y=True)


#### 可視化結果の例

上記のコードを実行することで、以下のような可視化結果が得られます。これらの図から、モデルの予測精度と誤差の性質を総合的に判断できます。

**実測値 vs 予測値の散布図**
![実測値 vs 予測値の散布図](images/session11/scatter_plot.png)

**Bland-Altman プロット**
![Bland-Altman プロット](images/session11/bland_altman_plot.png)

**残差のヒストグラムと箱ひげ図**
![残差のヒストグラムと箱ひげ図](images/session11/residuals_histbox.png)

### 3-4. 練習のまとめ

- `LinearRegression`, `SVR`, `RandomForestRegressor` の3種類を比較する
- `GridSearchCV` でパラメータを調整し、汎化性能を高める
- `Bland-Altman プロット` で予測誤差に系統的な偏りがないか確認する
- `RMSE`, `MAE`, `R2` だけでなく、可視化結果も総合的に判断する

---


## 4. 今日の振り返り

- 回帰は連続値予測のための手法
- MSE / RMSE / MAE / R2 が代表的な評価指標
- 予測値と実測値の可視化でモデルの挙動を確認する
- 線形回帰の係数から特徴量の影響を推定できる
